In [1]:
import os
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

import rasterio
from rasterio.transform import from_origin
import gc

In [ ]:
maize = pd.read_csv('../../dataset/region prediction/maize_clustered.csv')
wheat = pd.read_csv('../../dataset/region prediction/wheat_clustered.csv')

In [3]:
maize_cluster0 = maize[maize["Cluster"] == 0].copy()
print(maize_cluster0.shape)

(384682, 12)


In [4]:
wheat_cluster2 = wheat[wheat["Cluster"] == 2].copy()
print(wheat_cluster2.shape)

(370897, 12)


In [6]:
maize_cluster0

,CEC,Clay,SOC,Sand,Silt,lat,lon,pH,MAT,MAP,Elevation,Cluster
49782,19.3,35.2,1.99,32.1,32.7,45.118772,85.305550,8.2,9.6,117.0,301.0,0.0
49911,21.1,27.9,1.26,30.8,41.3,45.419420,122.935952,7.8,5.3,412.0,146.0,0.0
50344,19.6,32.8,3.00,34.4,32.8,45.010015,84.734142,8.2,9.6,122.0,287.0,0.0
50346,20.3,35.8,1.68,31.3,32.9,45.124196,85.360940,8.2,9.6,117.0,303.0,0.0
50877,19.5,31.1,2.02,35.3,33.6,45.111372,85.314569,8.2,9.6,118.0,303.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1697131,21.7,24.5,2.75,29.9,45.6,44.595095,82.550977,8.1,8.6,126.0,290.0,0.0
1697149,19.6,30.1,2.78,33.3,36.6,45.013846,84.681865,8.1,9.5,122.0,289.0,0.0
1697654,20.5,23.4,2.13,29.3,47.3,44.596966,82.581777,8.1,8.5,125.0,292.0,0.0
1697680,20.7,36.4,1.92,30.9,32.7,45.123009,85.304031,8.2,9.6,116.0,301.0,0.0


In [7]:
wheat_cluster2

,CEC,Clay,SOC,Sand,Silt,lat,lon,pH,MAT,MAP,Elevation,Cluster
9907,18.5,29.4,2.39,30.3,40.3,46.658163,83.269984,7.9,6.5,267.0,550.0,2.0
10930,19.5,26.2,2.03,30.9,42.8,46.250551,83.022076,8.0,6.2,253.0,623.0,2.0
10931,19.5,26.8,2.00,31.4,41.8,46.251768,83.028091,8.0,6.2,249.0,622.0,2.0
13617,23.7,27.1,1.40,40.1,32.8,45.725529,125.105743,8.0,4.3,453.0,135.0,2.0
13899,22.1,23.8,1.57,49.0,27.3,45.890273,124.003541,7.8,4.8,408.0,128.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
708425,22.2,25.5,1.02,26.6,47.9,34.018339,116.647120,8.5,15.1,748.0,36.0,2.0
708426,22.6,22.9,0.96,27.9,49.2,34.016810,116.663030,8.5,15.1,746.0,35.0,2.0
708427,21.8,26.9,0.84,24.6,48.6,34.014768,116.684242,8.5,15.1,746.0,35.0,2.0
708428,22.0,26.7,0.93,25.6,47.7,34.013745,116.694847,8.5,15.1,746.0,35.0,2.0


In [5]:
drop_cols = ['MAT','MAP','Elevation','Cluster']

In [6]:
maize_env = maize_cluster0.drop(columns=drop_cols)
wheat_env = wheat_cluster2.drop(columns=drop_cols)

In [7]:
model = CatBoostRegressor()
model.load_model("../../notebooks/Model/catboost_china_prediction.cbm")
with open("../../notebooks/Model/features_china_prediction.json", "r") as f:
    feature_names = json.load(f)

print("\nTotal features:", len(feature_names))


Total features: 44


In [8]:
crop_traits = {
    "maize": {
        "Family": "Poaceae",
        "Genus": "Zea",
        "Order": "Poales",
        "monocot": 1,
        "woody": 0,
        "herb": 1,
        "crop": 1,
        "vegetable": 0,
        "legume": 0,
        "grass": 1,
        "perennial": 0,
        "edible": 1,
        "logED": 2.176091259
    },

    "wheat": {
        "Family": "Poaceae",
        "Genus": "Triticum",
        "Order": "Poales",
        "monocot": 1,
        "woody": 0,
        "herb": 1,
        "crop": 1,
        "vegetable": 0,
        "legume": 0,
        "grass": 1,
        "perennial": 0,
        "edible": 1,
        "logED": 2.380211242
    }
}

In [9]:
pfas_table = pd.read_excel("../../dataset/PFAS.xlsx")
pfas_table['logIpc'] = np.log10(pfas_table['Ipc'])
print(pfas_table.shape)
print(pfas_table.head())

(41, 21)
  PFAS_name                                         Raw_SMILES  \
0      PFBA                     C(=O)(C(C(C(F)(F)F)(F)F)(F)F)O   
1     PFPeA              C(=O)(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)O   
2     PFHxA       C(=O)(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)O   
3     PFHpA  C(=O)(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(...   
4      PFOA  C(=O)(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F...   

   C-F chain length           Ipc  BCUT2D_MRLOW  BCUT2D_MWLOW        SPS  \
0                 3    260.658616     -0.347050     10.147392  14.384615   
1                 4    939.726831     -0.389910     10.044436  15.062500   
2                 5   3314.446802     -0.417659      9.980602  15.526316   
3                 6  11530.539061     -0.436369      9.938399  15.863636   
4                 7  39741.897337     -0.449718      9.909071  16.120000   

   FractionCSP3  VSA_EState7  MaxAbsEStateIndex  ...    Kappa3  HallKierAlpha  \
0      0.750000    -6.602292          11.750579  ...  1.

In [10]:
# ==========================================================
# Remove unnecessary columns
# ==========================================================

drop_cols = ["Raw_SMILES","C-F chain length",'Ipc']

pfas_table = pfas_table.drop(

    columns=drop_cols,

    errors="ignore"
)

In [11]:
# ==========================================================
# Convert descriptor columns to numeric
# ==========================================================

for col in pfas_table.columns:

    if col != "PFAS_name":

        pfas_table[col] = pd.to_numeric(

            pfas_table[col],

            errors="coerce"
        )

In [12]:
# ==========================================================
# Build PFAS_DICT
# ==========================================================

PFAS_DICT = {}

for _, row in pfas_table.iterrows():

    pfas_name = row["PFAS_name"]

    descriptor_dict = row.drop(

        labels=["PFAS_name"]

    ).to_dict()

    PFAS_DICT[pfas_name] = descriptor_dict

In [13]:
# ==========================================================
# Check
# ==========================================================

print("\nTotal PFAS:")

print(len(PFAS_DICT))

print("\nPFAS names:")

print(list(PFAS_DICT.keys())[:10])

print("\nExample:")

example_name = list(PFAS_DICT.keys())[0]

print(example_name)

print(PFAS_DICT[example_name])


Total PFAS:
41

PFAS names:
['PFBA', 'PFPeA', 'PFHxA', 'PFHpA', 'PFOA', 'PFNA', 'PFDA', 'PFUnDA', 'PFDoDA', 'PFTrDA']

Example:
PFBA
{'BCUT2D_MRLOW': -0.347050241178715, 'BCUT2D_MWLOW': 10.1473915373546, 'SPS': 14.3846153846154, 'FractionCSP3': 0.75, 'VSA_EState7': -6.60229166666667, 'MaxAbsEStateIndex': 11.7505787037037, 'qed': 0.712441776552488, 'Kappa3': 1.67669845728037, 'HallKierAlpha': -1.02, 'MinAbsEStateIndex': 3.53256944444444, 'MaxPartialCharge': 0.460094687009651, 'MinAbsPartialCharge': 0.460094687009651, 'MinPartialCharge': -0.476625664235965, 'EState_VSA9': 5.10652739484071, 'MaxAbsPartialCharge': 0.476625664235965, 'PEOE_VSA3': 4.79453718407182, 'logIpc': 2.41607208477418}


In [14]:
maize_SRC = {
    "PFBA":0.031117143,
    "PFPeA":0.024858571,
    "PFHxA":0.016937143,
    "PFHpA":0.013774286,
    "PFOA":0.320571429,
    "PFNA":0.0412,
    "PFDA":0.02014,
    "PFUnDA":0.015994286,
    "PFDoDA":0.002767143,
    "PFTrDA":0.003975714,
    "PFTeDA":0.000463571,
    "PFBS":0.008528571,
    "PFHxS":0.00001,
    "PFOS":0.013094286,
    "PFDS":0.000281429
}

wheat_SRC = {
    "PFBA":0.020134444,
    "PFPeA":0.021268889,
    "PFHxA":0.032223333,
    "PFHpA":0.017078889,
    "PFOA":0.161923333,
    "PFNA":0.131955556,
    "PFDA":0.01247,
    "PFUnDA":0.072088889,
    "PFDoDA":0.002505556,
    "PFTrDA":0.033871111,
    "PFTeDA":0.000603889,
    "PFBS":0.017081111,
    "PFHxS":0.00001,
    "PFOS":0.005995556,
    "PFDS":0.00001
}

In [15]:
# ==========================================================
# Build prediction dataset
# ==========================================================

def build_prediction_dataset(
    env_df,
    crop_name,
    pfas_name,
    tissue
):

    df = env_df.copy()

    # ======================================================
    # Add crop traits
    # ======================================================

    for k, v in crop_traits[crop_name].items():

        df[k] = v

    # ======================================================
    # Add PFAS descriptors
    # ======================================================

    for k, v in PFAS_DICT[pfas_name].items():

        df[k] = v

    # ======================================================
    # Add soil background concentration
    # ======================================================

    if crop_name == "maize":
        df["logSRC"] = np.log10(maize_SRC[pfas_name])
    else:
        df["logSRC"] = np.log10(wheat_SRC[pfas_name])

    # ======================================================
    # Add tissue
    # ======================================================

    df["Tissue"] = tissue

    # ======================================================
    # Fill missing columns
    # ======================================================

    for col in feature_names:

        if col not in df.columns:

            if col in [
                "Family",
                "Genus",
                "Order",
                "Tissue"
            ]:

                df[col] = "Unknown"

            else:

                df[col] = 0

    # ======================================================
    # Reorder
    # ======================================================

    df = df[feature_names]

    return df


In [16]:
print(maize_env[['lon','lat']].head())
print(maize_env['lon'].diff().value_counts().head())
print(maize_env['lat'].diff().value_counts().head())

              lon        lat
49782   85.305550  45.118772
49911  122.935952  45.419420
50344   84.734142  45.010015
50346   85.360940  45.124196
50877   85.314569  45.111372
lon
0.005523    2
0.005954    1
0.550744    1
0.005946    1
0.005978    1
Name: count, dtype: int64
lat
 0.001075    1
 0.300648    1
-0.409405    1
 0.114181    1
-0.012823    1
Name: count, dtype: int64


In [17]:
# 所有需要预测的PFAS
PREDICT_PFAS = sorted(
    set(maize_SRC.keys()) & set(wheat_SRC.keys())
)

print(PREDICT_PFAS)

['PFBA', 'PFBS', 'PFDA', 'PFDS', 'PFDoDA', 'PFHpA', 'PFHxA', 'PFHxS', 'PFNA', 'PFOA', 'PFOS', 'PFPeA', 'PFTeDA', 'PFTrDA', 'PFUnDA']


In [18]:
CROP_LIST = [
    "wheat",
    "maize"
]

TISSUE_LIST = [
    "root",
    "fruit"
]

In [19]:
def lnBCF_to_concentration(df, src):

    out = df.copy()

    # 如果你的模型是lnBCF
    out["PlantConc"] = np.exp(out["lnBCF_pred"]) * src

    return out

In [20]:
# ==========================================================
# dataframe -> GeoTIFF
# ==========================================================

def dataframe_to_tif(
        df,
        value_col,
        output_tif,
        resolution=0.05,
        nodata=-9999
):

    df = df.copy()

    # ======================================================
    # Quantize coordinates
    # ======================================================

    df["lon"] = (
        np.round(df["lon"] / resolution)
        .astype(np.int32)
        * resolution
    )

    df["lat"] = (
        np.round(df["lat"] / resolution)
        .astype(np.int32)
        * resolution
    )

    # ======================================================
    # Extent
    # ======================================================

    xmin = df["lon"].min()
    xmax = df["lon"].max()

    ymin = df["lat"].min()
    ymax = df["lat"].max()

    # ======================================================
    # Raster size
    # ======================================================

    width = int(
        round((xmax - xmin) / resolution)
    ) + 1

    height = int(
        round((ymax - ymin) / resolution)
    ) + 1

    # ======================================================
    # Empty raster
    # ======================================================

    raster = np.full(
        (height, width),
        nodata,
        dtype=np.float32
    )

    # ======================================================
    # Row / Col
    # ======================================================

    cols = (
        (df["lon"] - xmin)
        / resolution
    ).round().astype(np.int32)

    rows = (
        (ymax - df["lat"])
        / resolution
    ).round().astype(np.int32)

    # ======================================================
    # Fill raster
    # ======================================================

    raster[
        rows,
        cols
    ] = df[value_col].values

    # ======================================================
    # Transform
    # ======================================================

    transform = from_origin(
        xmin - resolution / 2,
        ymax + resolution / 2,
        resolution,
        resolution
    )

    # ======================================================
    # Save
    # ======================================================

    with rasterio.open(
        output_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=np.float32,
        crs="EPSG:4326",
        transform=transform,
        nodata=nodata,
        compress="lzw"
    ) as dst:

        dst.write(raster, 1)

    print(f"GeoTIFF saved: {output_tif}")

In [21]:
ENV_DICT = {
    "maize": maize_env,
    "wheat": wheat_env
}

for crop_name in CROP_LIST:

    env_df = ENV_DICT[crop_name]

    for tissue in TISSUE_LIST:

        for pfas_name in PREDICT_PFAS:

            print("=" * 60)
            print(crop_name, tissue, pfas_name)

            X = build_prediction_dataset(
                env_df,
                crop_name,
                pfas_name,
                tissue
            )

            pred = model.predict(X)

            result = env_df[["lon", "lat"]].copy()

            result["lnBCF_pred"] = pred.astype(np.float32)

            # Soil concentration
            if crop_name == "maize":
                src = maize_SRC[pfas_name]
            else:
                src = wheat_SRC[pfas_name]

            # Plant concentration
            result = lnBCF_to_concentration(
                result,
                src
            )

            pkl_file = (
                f"before_calibration/"
                f"{pfas_name}_{crop_name}_{tissue}.pkl"
            )

            csv_file = (
                f"before_calibration/csv/"
                f"{pfas_name}_{crop_name}_{tissue}.csv"
            )

            tif_file = (
                f"before_calibration/tif/"
                f"{pfas_name}_{crop_name}_{tissue}.tif"
            )

            # =====================================
            # Save
            # =====================================

            result.to_pickle(pkl_file)

            result.to_csv(
                csv_file,
                index=False,
                float_format="%.6f"
            )

            dataframe_to_tif(
                result,
                value_col="PlantConc",
                output_tif=tif_file,
                resolution=0.005
            )

            print(f"Saved: {tif_file}")

            del X
            del pred
            del result

            gc.collect()

print("Single PFAS prediction finished.")

wheat root PFBA
GeoTIFF saved: before_calibration/tif/PFBA_wheat_root.tif
Saved: before_calibration/tif/PFBA_wheat_root.tif
wheat root PFBS
GeoTIFF saved: before_calibration/tif/PFBS_wheat_root.tif
Saved: before_calibration/tif/PFBS_wheat_root.tif
wheat root PFDA
GeoTIFF saved: before_calibration/tif/PFDA_wheat_root.tif
Saved: before_calibration/tif/PFDA_wheat_root.tif
wheat root PFDS
GeoTIFF saved: before_calibration/tif/PFDS_wheat_root.tif
Saved: before_calibration/tif/PFDS_wheat_root.tif
wheat root PFDoDA
GeoTIFF saved: before_calibration/tif/PFDoDA_wheat_root.tif
Saved: before_calibration/tif/PFDoDA_wheat_root.tif
wheat root PFHpA
GeoTIFF saved: before_calibration/tif/PFHpA_wheat_root.tif
Saved: before_calibration/tif/PFHpA_wheat_root.tif
wheat root PFHxA
GeoTIFF saved: before_calibration/tif/PFHxA_wheat_root.tif
Saved: before_calibration/tif/PFHxA_wheat_root.tif
wheat root PFHxS
GeoTIFF saved: before_calibration/tif/PFHxS_wheat_root.tif
Saved: before_calibration/tif/PFHxS_wheat_ro

In [22]:
# # ==========================================================
# # Chain-length grouped lnBCF
# # ==========================================================

CHAIN_GROUPS = {

    "short": [
        "PFBA",
        "PFPeA",
        "PFHxA",
        "PFBS",
        "PFHxS"
    ],

    "medium": [
        "PFHpA",
        "PFOA",
        "PFNA",
        "PFOS"
    ],

    "long": [
        "PFDA",
        "PFUnDA",
        "PFDoDA",
        "PFTrDA",
        "PFTeDA",
        "PFDS"
    ]
}

In [23]:
for crop_name in CROP_LIST:

    for tissue in TISSUE_LIST:

        for group_name, pfas_list in CHAIN_GROUPS.items():

            print("=" * 60)
            print(group_name, crop_name, tissue)

            conc_stack = []

            coord_df = None

            for pfas_name in pfas_list:

                file = (
                    f"before_calibration/"
                    f"{pfas_name}_{crop_name}_{tissue}.pkl"
                )

                df = pd.read_pickle(file)

                if coord_df is None:

                    coord_df = df[
                        ["lon", "lat"]
                    ].copy()

                conc_stack.append(
                    df["PlantConc"].values
                )

            conc_stack = np.vstack(conc_stack)

            total_conc = np.sum(
                conc_stack,
                axis=0
            ).astype(np.float32)

            result_df = coord_df.copy()

            result_df["PlantConc"] = total_conc

            # =====================================
            # Output paths
            # =====================================

            out_pkl = (
                f"before_calibration/"
                f"{group_name}_{crop_name}_{tissue}.pkl"
            )

            out_csv = (
                f"before_calibration/group_csv/"
                f"{group_name}_{crop_name}_{tissue}.csv"
            )

            out_tif = (
                f"before_calibration/group_tif/"
                f"{group_name}_{crop_name}_{tissue}.tif"
            )

            # =====================================
            # Save
            # =====================================

            result_df.to_pickle(out_pkl)

            result_df.to_csv(
                out_csv,
                index=False,
                float_format="%.6f"
            )

            dataframe_to_tif(
                result_df,
                value_col="PlantConc",
                output_tif=out_tif,
                resolution=0.005
            )

            print(result_df["PlantConc"].describe())

            del conc_stack
            del total_conc
            del result_df

            gc.collect()

print("Before-calibration chain groups finished.")

short wheat root
GeoTIFF saved: before_calibration/group_tif/short_wheat_root.tif
count    370897.000000
mean          0.415028
std           0.076294
min           0.240128
25%           0.380515
50%           0.413337
75%           0.444679
max           1.275715
Name: PlantConc, dtype: float64
medium wheat root
GeoTIFF saved: before_calibration/group_tif/medium_wheat_root.tif
count    370897.000000
mean          0.895369
std           0.154272
min           0.452463
25%           0.789727
50%           0.865687
75%           1.012796
max           1.473719
Name: PlantConc, dtype: float64
long wheat root
GeoTIFF saved: before_calibration/group_tif/long_wheat_root.tif
count    370897.000000
mean          0.367230
std           0.057970
min           0.152664
25%           0.335205
50%           0.359126
75%           0.408657
max           0.571092
Name: PlantConc, dtype: float64
short wheat fruit
GeoTIFF saved: before_calibration/group_tif/short_wheat_fruit.tif
count    370897.000000